In [ ]:
"""
File Name: Automated_Stats.ipynb
Author: Evan Anderson
School: Boston College & ESADE
Project Overview: Uses raw TrackMan data to find pitching team leaders in multiple statistics, including xWOBA. Assists with the end of
season review of player performance.
"""

In [ ]:
#Imports necessary libraries.
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
import pandas as pd

#Loads the master CSV (in this code, the master CSV is omitted due to NECBL data rules) with every pitch of the Waves 2026 season (Excel utilized to create master CSV).
df_raw=proprietary_data
df_raw=df_raw[['Pitcher','PitcherId','PitcherTeam','PitchCall','KorBB','PlayResult','RelSpeed','SpinRate','ExitSpeed','Angle']]
df_raw=df_raw[~df_raw['Pitcher'].isin(['Plasschaert, Zach','Applefield, Kalen','Lippman, Hank','Kahn, Matthew'])]

#Filters and cleans the data sheet to only include pitches thrown by the Waves pitcher.
df=df_raw[df_raw['PitcherTeam']=='OCE_STA6'].copy()
df=df.dropna(subset=['Pitcher','PitcherId','PitchCall'])

In [ ]:
#Creates a data set for pitches where the at_bat came to an end.
df_result=df[(df['KorBB'].isin(['Strikeout','Walk'])|df['PitchCall'].isin(['InPlay','HitByPitch']))].copy()

#Creates and cleans a data set for pitches where the ball was put in play.
df_bip=df_result[df_result['PitchCall']=='InPlay'].copy()
df_bip=df_bip.dropna(subset=['ExitSpeed','Angle'])

In [ ]:
#Assigns weights for each possible outcome of a ball in play.
woba_weights={'Single':0.88,'Double':1.25,'Triple':1.50,'HomeRun':2.03,'Out':0.00,'FieldersChoice':0.00,'Error':0.00,'Sacrifice':0.00}
df_bip['wOBA_val']=df_bip['PlayResult'].map(woba_weights).fillna(0)

In [ ]:
#Assigns columns to be dependent and independent variables for the regression analysis, and runs said regression.
X=df_bip[['ExitSpeed','Angle']]
Y=df_bip['wOBA_val']

model=RandomForestRegressor(n_estimators=150,max_depth=5, min_samples_leaf=8,random_state=5)
df_bip['x_wOBA_val']=cross_val_predict(model,X,Y,cv=5)
r2_scores=cross_val_score(model,X,Y,cv=5,scoring='r2')
print(f"Mean 5 Fold Cross Validated R^: {r2_scores.mean():.3f}")

In [ ]:
#Merges the ball in play expected data back into the overall end of at bat data set.
df_result=df_result.merge(df_bip[['x_wOBA_val']], left_index=True, right_index=True, how='left')

In [ ]:
#Reassigns the weights to the different play results for the overall data set.
df_result['wOBA_val']=df_result['PlayResult'].map(woba_weights).fillna(0)

#Copies over the expected values.

#Fill in the wOBA values for the strikeouts, walks, and hit-by-pitches.
df_result.loc[df_result['KorBB']=='Walk','x_wOBA_val']=0.69
df_result.loc[df_result['KorBB']=='Walk','wOBA_val']=0.69

df_result.loc[df_result['PitchCall']=='HitByPitch','x_wOBA_val']=0.69
df_result.loc[df_result['PitchCall']=='HitByPitch','wOBA_val']=0.69


df_result.loc[df_result['KorBB']=='Strikeout','x_wOBA_val']=0.00
df_result.loc[df_result['KorBB']=='Strikeout','wOBA_val']=0.00

In [ ]:
#Create binary numbers for strikeouts and walks for the additional statistical aggregations.
df_result['is_k']=(df_result['KorBB']=='Strikeout').astype(int)
df_result['is_bb']=(df_result['KorBB']=='Walk').astype(int)

In [ ]:
#Creates important statistics that will assist in finding important pitching metrics.
pitch_stats=df_result.groupby('Pitcher').agg(total_pa=('Pitcher','count'),total_k=('is_k','sum'),total_bb=('is_bb','sum')).copy()

#Creates the statistics to calculate whiff rate.
df['is_swing']=df['PitchCall'].isin(['StrikeSwinging','FoulBallFieldable','FoulBallNotFieldable','InPlay']).astype(int)
df['is_whiff']=(df['PitchCall']=='StrikeSwinging').astype(int)
pitch_count=df.groupby('Pitcher').agg(total_swing=('is_swing','sum'),total_whiff=('is_whiff','sum'))

In [ ]:
#Creates the final stats used for my project.
pitch_leaders=df_result.groupby('Pitcher').agg(pitcher_wOBA=('wOBA_val','mean'),pitcher_xwOBA=('x_wOBA_val','mean'))
pitch_leaders['max_velo']=df.groupby('Pitcher').agg(max_velo=('RelSpeed','max'))
pitch_leaders['max_spin']=df.groupby('Pitcher').agg(max_velo=('SpinRate','max'))
pitch_leaders['total_pa']=pitch_stats['total_pa']
pitch_leaders['k_rate']=(pitch_stats['total_k']/pitch_stats['total_pa'])*100
pitch_leaders['bb_rate']=(pitch_stats['total_bb']/pitch_stats['total_pa'])*100
pitch_leaders['whiff_rate']=(pitch_count['total_whiff']/pitch_count['total_swing'])*100

#Rounds the stats to standard decimal places.
pitch_leaders=pitch_leaders.round({'pitcher_wOBA':3,'pitcher_xwOBA':3,'max_velo':1,'max_spin':0,'k_rate':1,'bb_rate':1,'whiff_rate':1})
pitch_leaders['max_spin']=pitch_leaders['max_spin'].astype(int)

#Uses a final threshold of 30 plate appearances for pitchers to join the leaderboards.
pitch_leaders=pitch_leaders[pitch_leaders['total_pa']>30]
pitch_leaders.sort_values(by='pitcher_wOBA')

In [ ]:
#pitcher wOBA leaders
print("wOBA leaders:")
display(pitch_leaders.sort_values(by='pitcher_wOBA',ascending=True)[['pitcher_wOBA']].head())
print()

#pitcher velocity leaders
print("Maximum velocity leaders:")
display(pitch_leaders.sort_values(by='max_velo',ascending=False)[['max_velo']].head())
print()

#pitcher spin rate leaders
print("Maximum spin rate leaders:")
display(pitch_leaders.sort_values(by='max_spin',ascending=False)[['max_spin']].head())
print()

#pitcher strikeout rate leaders
print("Strikeout rate leaders:")
display(pitch_leaders.sort_values(by='k_rate',ascending=False)[['k_rate']].head())
print()

#pitcher walk rate leaders
print("Walk rate leaders:")
display(pitch_leaders.sort_values(by='bb_rate',ascending=True)[['bb_rate']].head())
print()

#pitcher whiff rate leaders
print("Whiff rate leaders:")
display(pitch_leaders.sort_values(by='whiff_rate',ascending=False)[['whiff_rate']].head())
print()

#pitcher xwOBA leaders
print("Expected wOBA leaders:")
display(pitch_leaders.sort_values(by='pitcher_xwOBA',ascending=True)[['pitcher_xwOBA']].head())